# Clinical Text Preprocessing and Tokenization

This notebook prepares de-identified clinical notes for downstream NLP tasks such as classification, embeddings, and retrieval.

## Clinical preprocessing principles

- normalizes casing and removes whitespace-only tokens 
- preserves negation, dosage values, punctuation, and stop words for future modeling

In [1]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_sci_sm")
notes = pd.read_csv("../data/sample_clinical_notes.csv")
notes.head()

/Users/aryan/Desktop/Personal Projects/MedExplain/mednlp-env/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


,patient_id,note
0,1,Patient diagnosed with hypertension. Started o...
1,2,Patient with type 2 diabetes mellitus prescrib...
2,3,Continue atorvastatin 20 mg nightly for manage...
3,4,Patient prescribed levothyroxine for hypothyro...
4,5,Started sertraline to help manage symptoms of ...


In [2]:
def tokenize_clinical_note(note):
    """Return clinically meaningful tokens while preserving the original wording."""
    doc = nlp(note)
    return [token.text for token in doc if not token.is_space]

preprocessed_notes = notes.copy()
preprocessed_notes["tokens"] = preprocessed_notes["note"].apply(tokenize_clinical_note)
preprocessed_notes["normalized_note"] = preprocessed_notes["tokens"].apply(
    lambda tokens: " ".join(token.lower() for token in tokens)
)
preprocessed_notes["token_count"] = preprocessed_notes["tokens"].str.len()

preprocessed_notes[["patient_id", "tokens", "normalized_note", "token_count"]]

,patient_id,tokens,normalized_note,token_count
0,1,"[Patient, diagnosed, with, hypertension, ., St...",patient diagnosed with hypertension . started ...,12
1,2,"[Patient, with, type, 2, diabetes, mellitus, p...",patient with type 2 diabetes mellitus prescrib...,13
2,3,"[Continue, atorvastatin, 20, mg, nightly, for,...",continue atorvastatin 20 mg nightly for manage...,11
3,4,"[Patient, prescribed, levothyroxine, for, hypo...",patient prescribed levothyroxine for hypothyro...,7
4,5,"[Started, sertraline, to, help, manage, sympto...",started sertraline to help manage symptoms of ...,11


In [4]:
example_note = "Patient denies chest pain but reports nausea."
print(tokenize_clinical_note(example_note))

# keeping words such as 'denies' is important: removing them would reverse the clinical meaning.

['Patient', 'denies', 'chest', 'pain', 'but', 'reports', 'nausea', '.']
